In [2]:
import re
import pandas as pd
from underthesea import word_tokenize
import string

c:\Python Project\Sentiment Analysis for Shopee products\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_train = pd.read_csv('../data/data_origin/Train.csv')
df_test = pd.read_csv('../data/data_origin/Test.csv')
df_dev = pd.read_csv('../data/data_origin/Dev.csv')

In [4]:
df_train.columns

Index(['index', 'comment', 'n_star', 'date_time', 'label'], dtype='str')

In [5]:
df_train

,index,comment,n_star,date_time,label
0,0,Mới mua máy này Tại thegioididong thốt nốt cảm...,5,2 tuần trước,{CAMERA#Positive};{FEATURES#Positive};{BATTERY...
1,1,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,5,14/09/2019,{BATTERY#Negative};{GENERAL#Positive};{OTHERS};
2,2,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,3,17/08/2020,{FEATURES#Negative};
3,3,"Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...",3,29/02/2020,{FEATURES#Negative};{BATTERY#Neutral};{GENERAL...
4,4,"Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...",5,4/6/2020,{BATTERY#Positive};{PERFORMANCE#Positive};{SER...
...,...,...,...,...,...
7781,7781,8g. Cái đi đánh là mạng giật giật ko chịu nổi....,1,13/10/2019,{FEATURES#Negative};{BATTERY#Negative};{PERFOR...
7782,7782,Mua dk giảm 500k mà lỗi lòi ra hết treo màn hì...,1,5/5/2020,{FEATURES#Negative};{PERFORMANCE#Negative};{PR...
7783,7783,Máy Sài 3 tháng rồi rất OK.pin trâu khỏi nói S...,5,23/12/2019,{BATTERY#Positive};{PERFORMANCE#Positive};{GEN...
7784,7784,"Rất tiếc hàng realme ko có ốp lưng ngoài , nên...",3,20/04/2020,{PRICE#Negative};{GENERAL#Positive};{SER&ACC#N...


> Handle punctuation, handle whitespace, handle icons in strings

In [6]:
EMOJI_PATTERN = re.compile(
    "["
    u"\U0001F600-\U0001F64F"  # emoticons
    u"\U0001F300-\U0001F5FF"  # symbols & pictographs
    u"\U0001F680-\U0001F6FF"  # transport & map symbols
    u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
    u"\U00002500-\U00002BEF"  # box drawing/symbols
    u"\U00002702-\U000027B0"  # dingbats
    u"\U000024C2-\U0001F251"
    u"\U0001f926-\U0001f937"  # supplemental emojis
    u"\U00010000-\U0010ffff"  # all other extended unicode
    u"\u2640-\u2642"
    u"\u2600-\u2B55"
    u"\u200d"
    u"\u23cf"
    u"\u23e9"
    u"\u231a"
    u"\ufe0f"  # dingbats variation selector
    u"\u3030"
    "]+", flags=re.UNICODE
)

# Tạo bảng dịch dấu câu cơ bản
TRANSLATOR = str.maketrans('', '', string.punctuation)

def remove_punctuation_optimized(comment):
    # Tránh lỗi nếu comment bị rỗng (NaN/None)
    if not isinstance(comment, str):
        return ""
        
    # 1. Xóa Emoji VÀ các ký tự đặc biệt trước
    new_string = re.sub(EMOJI_PATTERN, '', comment)
    
    # 2. Xóa các dấu câu cơ bản (nhanh gọn nhẹ bằng C-backend của Python)
    new_string = new_string.translate(TRANSLATOR)
    
    # 3. Xóa các dấu câu "dị" thường gặp trên mạng (tuỳ chọn thêm)
    new_string = re.sub(r'[“”‘’…–—]', '', new_string)
    
    # 4. Xóa toàn bộ khoảng trắng thừa (tab, newline, double space...) Ở BƯỚC CUỐI CÙNG
    new_string = re.sub(r'\s+', ' ', new_string).strip()
    
    return new_string

In [7]:
df_train['comment'] = df_train['comment'].apply(lambda x: x.lower())
df_train['comment'] = df_train['comment'].apply(remove_punctuation_optimized)
df_dev['comment'] = df_dev['comment'].apply(lambda x: x.lower())
df_dev['comment'] = df_dev['comment'].apply(remove_punctuation_optimized)
df_test['comment'] = df_test['comment'].apply(lambda x: x.lower())
df_test['comment'] = df_test['comment'].apply(remove_punctuation_optimized)

>Handle Teencode

In [8]:
TEENCODE_DICT = {
    # Nhóm phủ định / Mức độ
    "ko": "không", "k": "không", "kh": "không", "khg": "không", "kg": "không", "khong": "không",
    "dc": "được", "đc": "được", "đk": "được",
    "r": "rồi", "rùi": "rồi", "roi": "rồi",
    "qúa": "quá", "qa": "quá",
    "vs": "với", "v": "vậy", "zậy": "vậy", "z": "vậy",
    "ntn": "như thế nào", "tn": "thế nào",
    
    # Nhóm sản phẩm / Dịch vụ (Đặc thù data điện thoại/mua sắm)
    "sp": "sản phẩm", "dt": "điện thoại", "đt": "điện thoại", "dthoai": "điện thoại",
    "nv": "nhân viên", "bh": "bảo hành", "tgdd": "thế giới di động", "cskh": "chăm sóc khách hàng",
    "cửa hg": "cửa hàng", "ch": "cửa hàng",
    "sd": "sử dụng", "sài": "xài", "s/d": "sử dụng",
    "km": "khuyến mãi", "kmai": "khuyến mãi",
    
    # Nhóm linh kiện / Đặc tính kỹ thuật
    "pin": "pin", "bin": "pin", "pjn": "pin",
    "wf": "wifi", "wfi": "wifi",
    "cam": "camera", "m.hình": "màn hình", "mh": "màn hình",
    
    # Nhóm cảm xúc / Đánh giá
    "ok": "tốt", "okie": "tốt", "oke": "tốt", "okela": "tốt", "okl": "tốt",
    "đc": "được", "ổn": "tốt", "perfect": "hoàn hảo", "good": "tốt",
    "tệ": "kém", "chán": "kém", "lag": "giật", "giựt": "giật"
}

def normalize_teencode(text):
    if not isinstance(text, str):
        return ""
    
    # Bước 1: Chuyển về chữ thường (để dễ đối chiếu với từ điển)
    text = text.lower()
    
    # Bước 2: Xử lý chữ kéo dài (Ví dụ: "đẹppppp" -> "đẹp", "quááááá" -> "quá")
    # Giải thích Regex: Tìm 3 ký tự giống nhau liên tiếp trở lên và thu gọn lại thành 1 ký tự
    text = re.sub(r'([A-Za-zĐđÁáÀàẢảÃãẠạĂăẮắẰằẲẳẴẵẶặÂâẤấẦầẨẩẪẫẬậÉéÈèẺẻẼẽẸẹÊêẾếỀềỂểỄễỆệÍíÌìỈỉĨĩỊịÓóÒòỎỏÕõỌọÔôỐốỒồỔổỖỗỘộƠơỚớỜờỞởỠỡỢợÚúÙùỦủŨũỤụƯưỨứỪừỬửỮữỰựÝýỲỳỶỷỸỹỴỵ])\1{2,}', r'\1', text)
    
    # Bước 3: Thay thế teencode dựa trên từ điển
    # Dùng split() để tách thành mảng các từ, tránh lỗi thay thế sai vào giữa từ chuẩn
    words = text.split()
    normalized_words = []
    
    for word in words:
        # Nếu từ có trong từ điển thì lấy giá trị chuẩn, không thì giữ nguyên
        std_word = TEENCODE_DICT.get(word, word)
        normalized_words.append(std_word)
        
    # Nối lại thành câu hoàn chỉnh
    text = " ".join(normalized_words)
    
    return text

#test hàm normalize_teencode với một số câu mẫu
sample_texts = [
    "Máy này sài rất ok, bin  trâu, nv tư vấn nhiệt tình quaáááá",
    "Dt bị lag, k bắt  dc wfi, chánnnnnn",
    "Mới mua hqua ở tgdd, sp tốt, camera đẹpppp"
]
sample_texts = [remove_punctuation_optimized(s) for s in sample_texts]
print("--- KẾT QUẢ TEST HÀM ---")
for s in sample_texts:
    print(f"Gốc: {s}")
    print(f"Sạch: {normalize_teencode(s)}\n")

--- KẾT QUẢ TEST HÀM ---
Gốc: Máy này sài rất ok bin trâu nv tư vấn nhiệt tình quaáááá
Sạch: máy này xài rất tốt pin trâu nhân viên tư vấn nhiệt tình quaá

Gốc: Dt bị lag k bắt dc wfi chánnnnnn
Sạch: điện thoại bị giật không bắt được wifi kém

Gốc: Mới mua hqua ở tgdd sp tốt camera đẹpppp
Sạch: mới mua hqua ở thế giới di động sản phẩm tốt camera đẹp



In [9]:
df_train['comment'] = df_train['comment'].apply(normalize_teencode)
df_dev['comment'] = df_dev['comment'].apply(normalize_teencode)
df_test['comment'] = df_test['comment'].apply(normalize_teencode)

>Word Segmentation

In [10]:
def segment_vietnamese(text):
    # Tránh lỗi sập code nếu gặp dòng rỗng (NaN/Float)
    if not isinstance(text, str) or text.strip() == "":
        return ""
    
    # format="text" là bắt buộc để tự động thêm dấu gạch dưới "_" cho PhoBERT
    return word_tokenize(text, format="text")

In [11]:
df_train["comment"] = df_train["comment"].apply(segment_vietnamese)
df_dev["comment"] = df_dev["comment"].apply(segment_vietnamese)
df_test["comment"] = df_test["comment"].apply(segment_vietnamese)

In [12]:
df_train.head()

,index,comment,n_star,date_time,label
0,0,mới mua máy này tại thegioididong thốt nốt cảm...,5,2 tuần trước,{CAMERA#Positive};{FEATURES#Positive};{BATTERY...
1,1,pin kém còn lại miễn_chê mua 832019 tình_trạng...,5,14/09/2019,{BATTERY#Negative};{GENERAL#Positive};{OTHERS};
2,2,sao lúc gọi điện_thoại màn_hình bị chấm nhỏ nh...,3,17/08/2020,{FEATURES#Negative};
3,3,mọi người cập_nhật phần_mềm lại nó sẽ bớt tốn ...,3,29/02/2020,{FEATURES#Negative};{BATTERY#Neutral};{GENERAL...
4,4,mới mua xài được 1 tháng thấy pin rất trâu xài...,5,4/6/2020,{BATTERY#Positive};{PERFORMANCE#Positive};{SER...


In [13]:
def map_sentiment_label(star):
    try:
        star = int(star)
        if star <= 2:
            return "negative"
        elif star == 3:
            return "neutral"
        else:
            return "positive"
    except:
        return None

def map_sentiment_id(label):
    # PhoBERT cần nhãn dạng số (Integer) bắt đầu từ 0
    mapping = {"negative": 0, "positive": 1, "neutral": 2}
    return mapping.get(label, -1)

# --- ÁP DỤNG VÀO DATAFRAME ---
print("Đang tạo nhãn Sentiment...")

# 1. Tạo cột nhãn dạng Text (Để con người dễ đọc/kiểm tra)
df_train['sentiment_label'] = df_train['n_star'].apply(map_sentiment_label)
df_test['sentiment_label'] = df_test['n_star'].apply(map_sentiment_label)
df_dev['sentiment_label'] = df_dev['n_star'].apply(map_sentiment_label)
# 2. Tạo cột nhãn dạng Số (Để đưa vào cho máy học)
df_train['label_id'] = df_train['sentiment_label'].apply(map_sentiment_id)
df_test['label_id'] = df_test['sentiment_label'].apply(map_sentiment_id)
df_dev['label_id'] = df_dev['sentiment_label'].apply(map_sentiment_id)

Đang tạo nhãn Sentiment...


In [16]:
df_train.to_csv('../data/data_preprocessed/Train_preprocessed.csv', index=False)
df_dev.to_csv('../data/data_preprocessed/Dev_preprocessed.csv', index=False)
df_test.to_csv('../data/data_preprocessed/Test_preprocessed.csv', index=False)